# H&M 데이터 전처리 및 클리닝 파이프라인
> **목적**: 대규모 트랜잭션 데이터를 정제하여 분석의 신뢰성을 확보하고 연령대별 파생 변수를 생성합니다.

### 주요 작업 내용:
1. **환경 설정**: 분석의 일관성을 위한 시각화 폰트 및 경고 제어
2. **데이터 진단**: 3,100만 건 데이터의 기술적 특성 파악
3. **결측치 처리**: 나이 및 범주형 데이터의 결측치 처리 (중앙값 활용)
4. **이상치 탐지**: Boxplot을 활용한 가격 데이터 분석
5. **데이터 타입 보존**: 처리 속도 최적화를 위한 Pickle 저장

Step 0. 환경 설정: 분석의 일관성을 위해 한글 폰트(Malgun Gothic) 설정 및 warnings 제어

In [ ]:
# Step 0. 라이브러리 임포트 및 시각화 설정

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 경고 무시 및 시각화 설정
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set(style="whitegrid")


Step 1. 데이터 로드: 약 3,100만 건의 트랜잭션 데이터를 로드하고 info()를 통해 데이터의 기술적 특성을 진단

In [ ]:
# Step 1. 데이터 로드 및 초기 구조 확인

print("--- [Step 1] 데이터 로드 시작 ---")
customers = pd.read_csv('customers.csv')
articles = pd.read_csv('articles.csv')
transactions = pd.read_csv('transactions_train.csv')

# 데이터 규모 확인
for name, df in zip(['Customers', 'Articles', 'Transactions'], [customers, articles, transactions]):
    print(f"{name} shape: {df.shape}")

# 상위 5행 및 요약 정보 확인
print("\n--- Customers 데이터 요약 ---")
print(customers.head())
print(customers.info())

Step 2. 결측치 처리: age 데이터의 분포를 고려하여 중앙값(Median)으로 대체하고, 범주형 데이터의 결측치를 'None'으로 명시하여 분석 결과의 왜곡을 방지

In [ ]:
# Step 2. 결측치(Missing Values) 확인 및 처리

print("\n--- [Step 2] 결측치 확인 및 처리 시작 ---")

# 2.1 결측치 개수 파악
print("\n고객 데이터 결측치 현황:")
print(customers.isnull().sum())

# 2.2 'age' 결측치 처리 (중앙값 대체)
# 분포 확인 후 왜곡을 최소화하기 위해 중앙값 선택 근거 제시
print(f"Age 중앙값: {customers['age'].median()}")
customers['age'] = customers['age'].fillna(customers['age'].median())

# 2.3 'fashion_news_frequency' 처리 (데이터 정합성 확보)
# 'NONE'과 결측치를 'None' 문자열로 통일
print(f"변경 전 카운트:\n{customers['fashion_news_frequency'].value_counts(dropna=False)}")
customers['fashion_news_frequency'] = customers['fashion_news_frequency'].replace({'NONE': 'None', np.nan: 'None'})

# 2.4 'club_member_status' 처리
customers['club_member_status'] = customers['club_member_status'].fillna('None')

print("\n전처리 후 결측치 확인:")
print(customers.isnull().sum())

Step 3. 이상치 탐지: price 컬럼의 Boxplot 분석을 통해 상위 0.1%의 고가 제품 거래 데이터를 식별

In [ ]:
# Step 3. 이상치(Outliers) 탐지 및 처리

print("\n--- [Step 3] 이상치 탐지 시작 ---")

# 3.1 거래 가격(price) 기술 통계 확인
print("\n가격 데이터 기술통계:")
print(transactions['price'].describe())

# 3.2 Boxplot을 이용한 시각적 이상치 확인
plt.figure(figsize=(10, 5))
sns.boxplot(x=transactions['price'])
plt.title('Transactions Price Boxplot (이상치 탐지)')
plt.show()

# 3.3 극단치(상위 0.1%) 임계값 확인
q_limit = transactions['price'].quantile(0.999)
print(f"상위 0.1% 임계값: {q_limit}")
# 도메인 특성상 고가 제품군으로 간주하여 유지

Step 4. 변수 생성: datetime 형식 변환 및 비즈니스 타겟인 연령대 그룹(age_group) 변수를 생성

In [ ]:
# Step 4. 데이터 타입 변환 및 파생 변수 생성

print("\n--- [Step 4] 데이터 타입 변환 및 파생 변수 생성 ---")

# 4.1 날짜 형식 변환
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

# 4.2 연령대 그룹화 (이력서 기준: 16-24, 25-34, 35+)
def get_age_group(age):
    if age <= 24: return '16-24'
    elif age <= 34: return '25-34'
    else: return '35+'

customers['age_group'] = customers['age'].apply(get_age_group)

Step 5. 데이터 병합: 분석 효율을 높이기 위해 필요한 컬럼만을 선별하여 최종 분석용 데이터셋을 구축

In [ ]:
# Step 5. 분석용 데이터셋 병합 및 저장

print("\n--- [Step 5] 데이터 병합 및 저장 ---")

# 분석 효율을 위해 필요한 컬럼만 병합
df_clean = transactions.merge(customers[['customer_id', 'age_group']], on='customer_id', how='left')
df_clean = df_clean.merge(articles[['article_id', 'prod_name', 'product_group_name']], on='article_id', how='left')

# 정제된 데이터셋 저장
df_clean.to_pickle('hm_cleaned_data.pkl')
print(f"최종 병합 데이터 Shape: {df_clean.shape}")
print("데이터 전처리 과정이 완료되었습니다.")